# Train Denoise ECG (haar_sym_lite) tren **Kaggle**

Ban chinh cho Kaggle Notebooks (GPU P100/T4 free, ~30h/tuan).

**Bat buoc truoc khi chay - mo panel Settings ben phai:**
1. **Accelerator -> GPU** (P100 hoac T4).
2. **Internet -> On** (de `wfdb` tai du lieu PhysioNet).

**Chuan bi du lieu code (Add Data):** bam **Add Data -> Upload** va upload file zip code (chua chu "Denoise"). Neu muon mang cac run da train tu Colab sang: tai thu muc `denoise_runs` tren Google Drive ve, nen lai, upload thanh 1 dataset nua.

Duong dan Kaggle: dataset upload nam o `/kaggle/input/...` (chi doc); noi ghi duoc la `/kaggle/working/`.

## 0. Kiem tra GPU

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chua bat GPU! Settings -> Accelerator -> GPU roi Run all lai.'

## 1. Nap code (tu tim trong /kaggle/input) va copy ra thu muc ghi duoc

In [ ]:
import glob, os, shutil, zipfile

DEST = '/kaggle/working/code'
shutil.rmtree(DEST, ignore_errors=True)

hit = glob.glob('/kaggle/input/**/configs/default.yaml', recursive=True)
if hit:
    src_root = os.path.dirname(os.path.dirname(hit[0]))
    shutil.copytree(src_root, DEST)
    print('Nap code tu thu muc da giai nen:', src_root)
else:
    zips = glob.glob('/kaggle/input/**/*Denoise*.zip', recursive=True)
    assert zips, 'Khong thay code trong /kaggle/input. Hay Add Data -> Upload zip code.'
    shutil.rmtree('/kaggle/working/_unz', ignore_errors=True)
    os.makedirs('/kaggle/working/_unz', exist_ok=True)
    shutil.copy(max(zips, key=os.path.getsize), '/kaggle/working/_unz/code.zip')
    while True:
        zz = glob.glob('/kaggle/working/_unz/**/*.zip', recursive=True)
        if not zz:
            break
        for zp in zz:
            out = zp[:-4]
            os.makedirs(out, exist_ok=True)
            try:
                with zipfile.ZipFile(zp) as z:
                    z.extractall(out)
            except zipfile.BadZipFile:
                pass
            os.remove(zp)
    cfgp = glob.glob('/kaggle/working/_unz/**/configs/default.yaml', recursive=True)
    assert cfgp, 'Khong tim thay configs/default.yaml sau khi giai nen.'
    src_root = os.path.dirname(os.path.dirname(cfgp[0]))
    shutil.copytree(src_root, DEST)
    print('Nap code tu zip:', src_root)

os.chdir(DEST)
print('PROJECT DIR =', DEST, '| co src/:', os.path.isdir('src'), '| co configs/:', os.path.isdir('configs'))

## 2. Cai thu vien

In [ ]:
!pip install -q -r requirements.txt

## 3. Tai du lieu PhysioNet (can Internet = On)

In [ ]:
!pip install -q wfdb
import wfdb
wfdb.dl_database('mitdb', '/kaggle/working/data/MITDB')
wfdb.dl_database('nstdb', '/kaggle/working/data/NSTDB')
print('Downloaded.')

## 4. Sua config cho Kaggle (duong dan + noi luu runs)

In [ ]:
import yaml
RUNS_DIR = '/kaggle/working/denoise_runs'
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['data']['dataset_root']   = '/kaggle/working/data'
cfg['data']['processed_root'] = '/kaggle/working/data/processed'
cfg['output_root']            = RUNS_DIR
cfg['train']['device']        = 'cuda'
# (Tuy chon) giam epoch cho nhanh: cfg['train']['epochs'] = 30
yaml.safe_dump(cfg, open('configs/default.yaml','w'), sort_keys=False)
print('output_root =', cfg['output_root'])

## 5. Tien xu ly (tao manifest.csv + segments .npz)

In [ ]:
!PYTHONPATH=. python -m src.prepare_data --config configs/default.yaml
import os
print('manifest:', os.path.exists('/kaggle/working/data/processed/manifest.csv'))

## 6. (Tuy chon) Mang cac run da train tu truoc sang
Neu ban da upload thu muc `denoise_runs` (tu Colab/Drive) thanh 1 dataset, cell nay copy cac run co checkpoint vao thu muc lam viec de **khong train lai**.

In [ ]:
import glob, os, shutil
RUNS_DIR = '/kaggle/working/denoise_runs'
os.makedirs(RUNS_DIR, exist_ok=True)

# Tim MOI run qua file checkpoint_best.pt (khong ken cau truc thu muc zip tu Drive)
n = 0
for ck in glob.glob('/kaggle/input/**/checkpoint_best.pt', recursive=True):
    r = os.path.dirname(ck)
    dst = os.path.join(RUNS_DIR, os.path.basename(r))
    if not os.path.exists(dst):
        shutil.copytree(r, dst); n += 1
print('Da mang sang', n, 'run tu lan train truoc')
print('Cac run hien co:', len(glob.glob(os.path.join(RUNS_DIR, '*'))))

## 7. TRAIN - chi chay run CON THIEU (resume-aware)
Cell tu kiem tra run nao da co checkpoint va chi train phan con thieu. Chay lai bao nhieu lan cung duoc - cai nao xong tu bo qua.

In [ ]:
import os
RUNS_DIR = '/kaggle/working/denoise_runs'

jobs = []
for D in [1,2,3,4,5]:
    jobs.append(dict(wavelet='haar', loss='mse', alpha=1.0, mid=D))
for W in ['haar','db2','db4','sym4','coif1','bior2.2']:
    jobs.append(dict(wavelet=W, loss='mse', alpha=1.0, mid=3))
for A in [0,0.2,0.4,0.5,0.6,0.8,1.0]:
    jobs.append(dict(wavelet='haar', loss='mixed', alpha=A, mid=3))
jobs.append(dict(wavelet='haar', loss='mse_haar', alpha=1.0, mid=3))

def run_name(j):
    return f"haar_sym_lite_wav-{j['wavelet']}_base-32_exp-4_mid-{j['mid']}_loss-{j['loss']}_alpha-{float(j['alpha'])}"

seen=set(); uniq=[]
for j in jobs:
    n=run_name(j)
    if n not in seen:
        seen.add(n); uniq.append(j)

todo=[j for j in uniq if not os.path.exists(os.path.join(RUNS_DIR, run_name(j), 'checkpoint_best.pt'))]
print(f"Tong: {len(uniq)} | Da xong: {len(uniq)-len(todo)} | Con thieu: {len(todo)}")
for j in todo: print("  TODO:", run_name(j))

for j in todo:
    cmd = (f"python -m src.train --config configs/default.yaml --model haar_sym_lite "
           f"--wavelet {j['wavelet']} --loss {j['loss']} --alpha {j['alpha']} --mid_depth {j['mid']}")
    print("\n>>>", cmd); os.system(cmd)

## 8. (Tuy chon) Train 5 baseline de so sanh

In [ ]:
!bash scripts/train_baselines.sh

## 9. Evaluate + tong hop

In [ ]:
RUNS_DIR = '/kaggle/working/denoise_runs'
!python -m src.eval --config configs/default.yaml --run_glob "{RUNS_DIR}/*"
!python scripts/summarize_runs.py --run_glob "{RUNS_DIR}/*" --out "{RUNS_DIR}/summary_all_models.csv" 

## 10. TRA LOI: depth / wavelet / loss nao tot nhat?

In [ ]:
import pandas as pd
RUNS_DIR = '/kaggle/working/denoise_runs'
s = pd.read_csv(f'{RUNS_DIR}/summary_all_models.csv')
hs = s[s.model_name=='haar_sym_lite'].copy()

print('==== DEPTH tot nhat ====')
print(hs[(hs.loss=='mse') & (hs.wavelet=='haar')].sort_values('snr_imp', ascending=False)[['mid_depth','snr_imp','prd','rmse','cosine']].to_string(index=False))
print('\n==== WAVELET tot nhat ====')
print(hs[(hs.loss=='mse') & (hs.mid_depth==3)].sort_values('snr_imp', ascending=False)[['wavelet','snr_imp','prd','rmse','cosine']].to_string(index=False))
print('\n==== LOSS / ALPHA tot nhat ====')
print(hs[(hs.wavelet=='haar') & (hs.mid_depth==3)].sort_values('snr_imp', ascending=False)[['loss','alpha','snr_imp','prd','rmse','cosine']].to_string(index=False))

## 11. Train MODEL TOI UU (dien winner o muc 10)

In [ ]:
BEST_DEPTH   = 3
BEST_WAVELET = 'haar'
BEST_LOSS    = 'mse'
BEST_ALPHA   = 1.0
!python -m src.train --config configs/default.yaml --model haar_sym_lite \
    --wavelet {BEST_WAVELET} --loss {BEST_LOSS} --alpha {BEST_ALPHA} --mid_depth {BEST_DEPTH} --run_suffix optimal

## 12. Bao cao params + latency

In [ ]:
RUNS_DIR = '/kaggle/working/denoise_runs'
!PYTHONPATH=. python scripts/report_model_complexity.py \
    --models dw_cnn dw_se dnn_dan fcn liwave haar_sym_lite \
    --wavelets haar --latency --out "{RUNS_DIR}/model_complexity_report.csv" 

## 13. Ve hinh (metric + waveform)

In [ ]:
RUNS_DIR = '/kaggle/working/denoise_runs'
!python scripts/merge_noise_snr_summary.py --run_glob "{RUNS_DIR}/*" --out "{RUNS_DIR}/all_models_by_noise_snr.csv"
!PYTHONPATH=. python scripts/plot_results.py --summary "{RUNS_DIR}/summary_all_models.csv" --noise_snr "{RUNS_DIR}/all_models_by_noise_snr.csv" --out_dir "{RUNS_DIR}/figures/metrics"
!python -m src.visualize --config configs/default.yaml --run_glob "{RUNS_DIR}/*optimal*" --segment_index 0
from IPython.display import Image, display
import glob
for p in (sorted(glob.glob(f'{RUNS_DIR}/figures/metrics/*.png')) + sorted(glob.glob(f'{RUNS_DIR}/*optimal*/figures/waveforms/*.png')))[:8]:
    print(p); display(Image(p))

## 14. Mean+/-std + kiem dinh thong ke (optimal vs baseline tot nhat)

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon
RUNS_DIR = '/kaggle/working/denoise_runs'
!python scripts/merge_all_segment_csv.py --run_glob "{RUNS_DIR}/*" --out "{RUNS_DIR}/all_segment_metrics.csv"
seg = pd.read_csv(f'{RUNS_DIR}/all_segment_metrics.csv')
metrics = ['snr_imp','prd','rmse','mae','cosine']
def mean_std(g): return pd.Series({m: f"{g[m].mean():.4f} +/- {g[m].std():.4f}" for m in metrics})
print(seg.groupby('run_name').apply(mean_std).to_string())

mean_snr = seg.groupby('run_name')['snr_imp'].mean()
opt = [r for r in mean_snr.index if 'optimal' in r]
opt_run = opt[0] if opt else mean_snr.idxmax()
base_runs = seg[seg.model_name.isin(['dw_cnn','dw_se','dnn_dan','fcn','liwave'])].groupby('run_name')['snr_imp'].mean()
best_base = base_runs.idxmax()
print('\nOptimal:', opt_run, '\nBest baseline:', best_base)
key=['record_id','segment_id','noise_type','noise_snr']
a=seg[seg.run_name==opt_run].set_index(key)['snr_imp']; b=seg[seg.run_name==best_base].set_index(key)['snr_imp']
c=a.index.intersection(b.index)
st,p=wilcoxon(a.loc[c].values,b.loc[c].values)
print(f'Wilcoxon n={len(c)}, p={p:.3e}', '=> co y nghia' if p<0.05 else '=> chua co y nghia')

## 15. LUU KET QUA (Kaggle khong tu giu /kaggle/working)
Khi ngat session, `/kaggle/working` se mat neu chua luu. 2 cach giu `denoise_runs`:

1. **Save Version** (nut goc tren phai -> Save Version): toan bo `/kaggle/working` duoc luu thanh output cua version, va co the dung lai lam input cho lan sau (chinh la dataset de muc 6 mang sang).
2. **Nen & tai ve** bang cell duoi roi download tu tab Output.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/denoise_runs_backup', 'zip', '/kaggle/working/denoise_runs')
print('Da nen: /kaggle/working/denoise_runs_backup.zip -> tai ve o tab Output')